In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"

In [2]:
import random, numpy as np, torch
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.cuda.manual_seed_all(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
import yaml
from types import SimpleNamespace

def load_config(path):
    with open(path, "r") as f:
        cfg_dict = yaml.safe_load(f)
    return SimpleNamespace(**cfg_dict)

training_config = load_config("config/training_config.yaml")

In [4]:
# TODO: S3E pretraining workflow
from preprocessors.factory import load_preprocessor_model
preprocessor_config = load_config("config/preprocessor_config.yaml")
preprocessor_model = load_preprocessor_model(preprocessor_config)

from data.data_factory import data_provider
preprocessor_set, preprocessor_loader = data_provider(preprocessor_config)

from trainer.preprocessor import PreprocessorTrainer
preprocessor_trainer = PreprocessorTrainer(
    preprocessor_model=preprocessor_model,
    preprocessor_set=preprocessor_set,
    preprocessor_loader=preprocessor_loader,
    training_config=preprocessor_config
)
preprocessor_trainer.pretrain()

TypeError: types.SimpleNamespace() argument after ** must be a mapping, not NoneType

In [5]:
from data.data_factory import data_provider

train_set, train_loader = data_provider(training_config, 'train')
val_set, val_loader = data_provider(training_config, 'val')
test_set, test_loader = data_provider(training_config, 'test')
train_val_set, train_val_loader = data_provider(training_config, 'train_val')

Stock IDs below threshold (0.90): ['6669']
Stock IDs below threshold (0.90): ['6669']
Stock IDs below threshold (0.90): ['6669']
Stock IDs below threshold (0.90): ['6669']


In [6]:

from models.factory import load_main_model

training_config.num_stocks = test_set.get_num_stocks()
model = load_main_model(training_config, preprocessor_model=None)
# model = load_main_model(training_config, preprocessor_model=preprocessor_model)

TypeError: load_main_model() got an unexpected keyword argument 'preprocessor_model'

In [ ]:
from trainer.stock import Trainer

if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    raise ValueError("No GPU available")

trainer = Trainer(
    model=model,
    args=training_config,
    train_set=train_set,
    val_set=val_set,
    test_set=test_set,
    train_val_set=train_val_set,
    train_loader=train_loader,
    val_loader=val_loader,
    test_loader=test_loader,
    train_val_loader=train_val_loader,
    device=device
)

best_epoch = trainer.train_with_early_stop()
trainer.train_final(best_epoch)

In [ ]:
import pandas as pd
test_year = f"{pd.to_datetime(training_config.split_dates[2]).year}"
checkpoint_path = f"checkpoints/{training_config.model_name}_{training_config.category}_{test_year}_{training_config.loss}.pt"
trainer.save_checkpoint(checkpoint_path)

In [ ]:
trainer.evaluate_and_save(f"results/{training_config.result_file_name}")